*   Softmax regression is linear regression's classification counterpart.
*   It predicts a probability distribution over classes instead of one numerical value.

In [4]:
import math
import random
import numpy as np
import torch

# 4.1.1 Classification

## 1. Intuition

*   Classification means predicting a category. A category is a named group, such as `cat`, `dog`, or `shirt`.
> *   A class is one possible category.
> *   A label is the correct class for one example.
> *   A score, also called a logit, is a raw number a model assigns to a class before turning scores into probabilities.







## 2. Why this exists

*   Many ML tasks ask for a decision among choices rather than a number.
*   The model must compare classes, not just output one scalar prediction.

## 3. Examples

*   Manual Python: choose the class with the largest score.

In [5]:
classes = ["cat", "dog", "car"]
scores = [1.2, 3.4, 0.5]
best_index = scores.index(max(scores))
predicted_class = classes[best_index]
predicted_class # dog since it has the maximum score, or clases[1] where 3.4 is positioned under score[1]

'dog'

*   PyTorch: compute class scores from features and weights.

In [6]:
X = torch.tensor([[1.0, 2.0]]) # shape (1, 2)
W = torch.tensor([[0.1, 0.2, -0.3], [0.4, -0.5, 0.6]]) # shape (2, 3)
b = torch.tensor([[0.0, 0.1, -0.1]]) # shape (3, )
logits = X @ W + b # X @ W = shape (1, 3), or [1*0.1+2*0.4, 1*0.2-0.5*2, 1*-0.3+2*0.6], or [0.9, -0.8, 0.9] + [0.0, 0.1, -0.1] = [0.9, -0.7, 0.8]
logits

tensor([[ 0.9000, -0.7000,  0.8000]])

## 4. Step-by-step breakdown

*   The manual example compares three class scores.
> *   `max(scores)` finds the largest score.
> *   `scores.index(...)` finds the position of that largest score.
*   In the tensor example, `X` has one example with two features.
> *   `W` maps two input features to three class scores.
> *   `X @ W + b` returns one score per class.

## 5. Connection to ML systems

*   Classification models usually output one score per class.
*   Later, softmax turns those scores into probabilities that sum to 1.

## 6. Common confusion points

- A class score is not automatically a probability.
- The largest score determines the predicted class.
- The number of output scores should match the number of classes.
- Classification labels are categories, even when represented by integer IDs.

# 4.1.2 Loss Function

## 1. Intuition

*   Softmax turns raw class scores into probabilities. A probability is a number between 0 and 1, and class probabilities should sum to 1.


*   Cross-entropy loss penalizes the model when it assigns low probability to the correct class.
> *   Cross-entropy loss is computed as $\mathrm{Loss} = -\log(p_{\mathrm{correct}})$, where $p_{\mathrm{correct}}$ is the probability assigned to the **correct class**.

    For example:

    | Probability of Correct Class | Loss|
    |:-----------------------------------------------------:|-------------------------------------:|
    | 1.00 | 0.000 |
    | 0.90 | 0.105 |
    | 0.50 | 0.693 |
    | 0.10 | 2.303 |
    | 0.01 | 4.605 |

    **Key idea:**

    - A **high probability** for the correct class results in a **small loss**.
    - A **low probability** for the correct class results in a **large loss**.

    During training, the optimizer minimizes this loss, encouraging the model to assign higher probabilities to the correct class.

## 2. Why this exists

*   Classification training needs a loss that rewards probability placed on the correct class and penalizes probability placed elsewhere.

## 3. Examples

*   Manual softmax for three class scores.
*   Softmax uses Euler's number (`e`) because the exponential function has the property that its growth rate is proportional to its current value.
> *   This makes it useful for transforming raw scores into positive values while preserving their relative ordering.
> *   We can divide each value by the total to obtain fractional representations/probabilities that sum to 1.

In [7]:
scores = torch.tensor([1.0, 2.0, 0.0])
exp_scores = torch.exp(scores) # [e^1, e^2, e^0] = 11.1074
probs = exp_scores / exp_scores.sum() # [e^1/11.1074, e^2/11.1074, e^0/11.1074] as exp_scores.sum() got broadcasted
probs

tensor([0.2447, 0.6652, 0.0900])

*   Cross-entropy for the correct class index.

In [8]:
correct_class = 1
loss = -torch.log(probs[correct_class]) # log(probs[1] or -log(0.6652)
loss

tensor(0.4076)

## 4. Step-by-step breakdown

* `torch.exp(scores)` makes all values positive via the exponent to Euler's number.

* Dividing by `exp_scores.sum()` makes the values sum to 1.

* `correct_class = 1` means the second class is the true label.

* `probs[correct_class]` selects the probability assigned to the true class.

* `-torch.log(...)` gives a small loss for high probability and a large loss for low probability.

## Loss Functions: MSE vs Cross-Entropy

A **loss function** measures how wrong a model's prediction is. During training, the model adjusts its weights to minimize this loss.

  ---

  ## Mean Squared Error (MSE)

  MSE asks:

  > "How close is my prediction to the actual value?"

  It measures the distance between the predicted value and the true value.

  $$
  \mathrm{MSE} = \frac{1}{n}\sum(y-\hat{y})^2
  $$

  where:

  - `y` = actual value
  - `y_hat` = predicted value
  - `n` = number of predictions

  Example:

  Predicting a house price:

  ```
  Prediction: $450,000
  Actual:    $500,000
  ```

  The model made a numeric error:

  ```
  Error = -$50,000
  ```

  MSE squares this error, meaning larger mistakes receive larger penalties.

  Common MSE tasks:

  - House price prediction
  - Temperature prediction
  - Age prediction
  - Distance estimation
  - Image reconstruction

  These are **regression problems** because the output is a measurable quantity.

  ---

  # Cross-Entropy Loss

  Cross-entropy asks:

  > "How much probability did the model assign to the correct class?"

  It is used for **classification problems**, where the model chooses between categories.

  Example:

  ```
  Classes:

  0 = Cat
  1 = Dog
  2 = Bird
  ```

  The numbers are only labels. They do not represent quantities.

  The model outputs probabilities:

  ```
  Cat:  0.20
  Dog:  0.70
  Bird: 0.10
  ```

  If the correct answer is Dog:

  ```
  correct_class = 1
  ```

  we select:

  ```
  probability of correct class = 0.70
  ```

  The cross-entropy loss is:

  $$
  \mathrm{Loss} = -\log(p_{\mathrm{correct}})
  $$

  The idea:

  - High probability for the correct class → small loss
  - Low probability for the correct class → large loss

  Examples:

  | Probability of Correct Class | Loss |
  |-----------------------------|------|
  | 0.99 | Very small |
  | 0.90 | Small |
  | 0.50 | Medium |
  | 0.10 | Large |
  | 0.01 | Very large |

  Cross-entropy strongly punishes models that are confidently wrong.

  ---

  # Why Use Cross-Entropy Instead of MSE for Classification?

  MSE can technically be used for classification, but cross-entropy is usually better.

  Suppose two models predict the correct class:

  ```
  Model A:
  Correct class probability = 0.90

  Model B:
  Correct class probability = 0.10
  ```

  MSE will penalize both, but cross-entropy creates a much stronger penalty for the model that was confidently wrong.

  Cross-entropy encourages the model to:

  - Increase probability for the correct class
  - Decrease probability for incorrect classes
  - Produce probabilities that sum to 1

  ---

  # Key Difference

  | Loss Function | Used For | Main Question |
  |---|---|---|
  | MSE | Regression | How close is my number? |
  | Cross-Entropy | Classification | How confident was I in the correct category? |

  ---

  # Mental Model

  ```
  Neural Network Output
            |
            |
    -------------------
    |                 |
  Numbers          Categories
    |                 |
  Regression     Classification
    |                 |
    MSE       Cross-Entropy

  "How close?"  "How confident?"
  ```

  MSE is for predicting **amounts**.

  Cross-entropy is for predicting **choices between categories**.

## 5. Connection to ML systems

*   PyTorch usually combines softmax and cross-entropy in `torch.nn.CrossEntropyLoss` for numerical stability.
*   Numerical stability means avoiding avoidable overflow, underflow, or precision problems.

## 6. Common confusion points

- Softmax probabilities sum to 1 across classes.
- Cross-entropy uses the correct class probability.
- A confident wrong prediction gets a large loss.
- In PyTorch, `CrossEntropyLoss` expects raw logits, not already-softmaxed probabilities.
> * `CrossEntropyLoss` applies softmax to convert the logits into probabilities.
> * It then computes the cross-entropy loss using the probability assigned to the correct class.
> * So do not apply `softmax` before passing the outputs to `CrossEntropyLoss`.

# 4.1.3 Information Theory Basics

## 1. Intuition

*   Information theory studies uncertainty and information.


*   Surprisal measures how unexpected an event is.
> * A high-probability event is not surprising.
> * A low-probability event is surprising.

*   The negative logarithm, `-log(p)`, is a common mathematical measure of surprisal.



## 2. Why this exists

*   Cross-entropy loss is easier to understand when viewed as surprisal:
> * The model is penalized by how surprised it is when the true class occurs.
> * This is because surprise is calculated under `-log(p)`.
> * High confidence in the correct answer → little surprise → small loss.
> * Low confidence in the correct answer → lots of surprise → large loss.

    | Probability Assigned to Correct Class | Surprise (`-log(p)`) | Loss |
    |---------------------------------------|----------------------|------|
    | 1.00                                  | 0                    | None |
    | 0.90                                  | Small                | Small |
    | 0.50                                  | Moderate             | Moderate |
    | 0.10                                  | Large                | Large |
    | 0.01                                  | Very large           | Very large |

## 3. Examples

*   Compare surprisal for high and low probabilities.

In [9]:
high_prob = torch.tensor(0.9)
low_prob = torch.tensor(0.1)
high_surprisal = -torch.log(high_prob)
low_surprisal = -torch.log(low_prob)
high_surprisal, low_surprisal

(tensor(0.1054), tensor(2.3026))

## 4. Step-by-step breakdown

*   `high_prob` represents a confident probability for the event.

*   `low_prob` represents an unlikely event.

*   `-log(0.9)` is small because the event is not surprising.

*   `-log(0.1)` is larger because the event is surprising.

*   Cross-entropy applies this idea to the true class probability.

## 5. Connection to ML systems

*   Classification training minimizes average surprisal of the correct labels under the model's predicted probabilities.

## 6. Common confusion points

- Information theory terms can sound abstract, but here the key idea is surprise.
- Logarithms turn probability products into sums, which is useful for optimization.
- Cross-entropy is not accuracy; it uses probability confidence.
- A model can have good accuracy but poor cross-entropy if it is badly calibrated.



    | Model | Accuracy | Typical Prediction |
    |-------|---------:|--------------------|
    | C | 90% | Usually predicts 0.999 confidence, but is wrong 10% of the time |

  - Model C has **90% accuracy**, but when it's wrong, it's **extremely confident**.

  - Those few mistakes produce **very large cross-entropy losses**, because:
  - Correct prediction with **0.999** confidence → **tiny loss**
  - Wrong prediction with **0.999** confidence → **huge loss**

  - So despite having the **same accuracy**, Model C can have **much worse cross-entropy** because cross-entropy penalizes **confident mistakes** much more heavily than less confident ones.

# 4.1.4 Summary and Discussion

## 1. Intuition

*   Softmax regression maps input features to one score per class, converts scores to probabilities, and trains with cross-entropy loss.

## 2. Why this exists

*   It is the simplest linear classifier and the foundation for later neural classifiers.

## 3. Examples

*   One compact softmax-regression forward pass.

In [10]:
X = torch.tensor([[1.0, 2.0], [0.5, -1.0]])
W = torch.randn(2, 3)
b = torch.randn(3)

logits = X @ W + b # logits.shape = (2,3) because (2,2) @ (2,3) = (2,3)
probs = torch.softmax(logits, dim=1)
probs.shape, probs.sum(dim=1) # probs.shape (2,3) as each row is a probability distribution over the 3 classes; probs.sum(dim=1) sums across the 3 classes for each sample, verifying that each row sums to 1.

(torch.Size([2, 3]), tensor([1.0000, 1.0000]))

## 4. Step-by-step breakdown

*   `X @ W + b` computes raw scores for each example and class.

*   `torch.softmax(logits, dim=1)` normalizes across class columns.

*   `probs.shape` is `(2, 3)`: two examples and three classes.

*   `probs.sum(dim=1)` checks that each example's class probabilities sum to 1.

## 5. Connection to ML systems

*   Deep classifiers often end with the same pattern: class logits followed by cross-entropy loss.

## 6. Common confusion points

- Use `dim=1` when rows are examples and columns are classes.
- Softmax is about relative scores, not absolute score size alone.
- Cross-entropy trains probabilities, not only the final class choice.
- The model is still linear unless hidden layers are added.

# 4.1.5 Exercises

## 1. Intuition

*   Exercises should make you inspect logits, probabilities, labels, and loss separately.

## 2. Why this exists

*   Classification bugs often come from mixing up class dimension, label format, or whether values are logits or probabilities.

## 3. Examples

*   Exercise 1: turn logits into probabilities.

In [11]:
logits = torch.tensor([[2.0, 1.0, 0.0]])
probs = torch.softmax(logits, dim=1)
probs, probs.sum(dim=1) # e^2+e^1+e^0 = 11.107; probs = [e^2/11.107, e^1/11.107, e^0/11.107], probs.sum(dim=1)=[1]

(tensor([[0.6652, 0.2447, 0.0900]]), tensor([1.]))

*   Exercise 2: compute cross-entropy with PyTorch from raw logits.

In [12]:
labels = torch.tensor([0])
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(logits, labels)
loss

# The correct class is class 0 (from labels = [0]).
# The logit for class 0 is 2.0.

# Softmax probability for class 0:
# e^2 / (e^2 + e^1 + e^0) = 7.389 / 11.107 = 0.6652

# CrossEntropyLoss uses:
# loss = -log(probability of the correct class)
# loss = -log(0.6652) ≈ 0.4076

tensor(0.4076)

## 4. Step-by-step breakdown

*   Exercise 1 checks softmax normalization across classes.

*   Exercise 2 checks the PyTorch convention: pass raw logits and integer class labels.

*   The label `0` means the first class is correct.

## 5. Connection to ML systems

*   These exercises prepare for from-scratch and concise softmax-regression implementations.

## 6. Common confusion points

- Do not pass one-hot labels to `CrossEntropyLoss` in the basic PyTorch use case.
- Do not apply softmax before `CrossEntropyLoss` unless you know the alternative API expects it.
- Check the class dimension before using softmax.
- A probability vector should sum to 1 across classes.

## One-hot labels (do not use in the basic case)

A one-hot label represents the class with a vector:

```python
[1, 0, 0]  # class 0
[0, 1, 0]  # class 1
[0, 0, 1]  # class 2
```

Example:

```python
labels = torch.tensor([[1, 0, 0]])
```

Shape:

```python
(1, 3)
```

This is **not the standard input format** for `CrossEntropyLoss`.

The basic PyTorch usage is:

```python
logits = model(x)          # shape: (batch_size, num_classes)
labels = torch.tensor([0]) # shape: (batch_size)

loss = CrossEntropyLoss()(logits, labels)
```

`CrossEntropyLoss` expects:
- **Logits:** raw model outputs with shape `(batch_size, num_classes)`
- **Labels:** class indices with shape `(batch_size,)`

Example:

```python
logits = torch.tensor([[2.0, 1.0, 0.0]])

labels = torch.tensor([0])
# means the correct class is class 0
```

The label `0` refers to the **index of the correct class**, not the probability or the logit value.

## Why mention "alternative API"?

Some loss functions **do expect probabilities or log probabilities**.

For example:

```python
torch.nn.NLLLoss()
```

expects **log probabilities**, so you might use:

```python
log_probs = torch.log_softmax(logits, dim=1)

loss = torch.nn.NLLLoss()(log_probs, labels)
```

The workflow is:

```text
logits
  |
  ↓
log_softmax()
  |
  ↓
log probabilities
  |
  ↓
NLLLoss()
  |
  ↓
loss
```

However, with:

```python
torch.nn.CrossEntropyLoss()
```

you give it **raw logits**:

```python
loss = torch.nn.CrossEntropyLoss()(logits, labels)
```

The workflow is:

```text
logits
  |
  ↓
CrossEntropyLoss()
  |
  ↓
(softmax + log + negative log likelihood internally)
  |
  ↓
loss
```

So:

- `CrossEntropyLoss` → pass **raw logits** ✅
- `NLLLoss` → pass **log probabilities** (usually from `log_softmax`) ✅

Do not apply `softmax` before `CrossEntropyLoss`, because it already handles that internally.

## The rule for summing dimensions

Do not ask:

> "Which dimension is last?"

Ask:

> "Which dimension contains the alternatives that should become probabilities?"

For classification:

```text
(batch, classes)
          ↑
       softmax here
```

For sequence classification:

```text
(batch, sequence, classes)
                    ↑
                 softmax here
```

For image segmentation:

```text
(batch, classes, height, width)
          ↑
       softmax here
```

---

### Practical PyTorch habit

Many models use the convention that the **class dimension is last**, so this is common:

```python
torch.softmax(logits, dim=-1)
```

`dim=-1` means:

> Apply softmax over the last dimension.

For example:

```python
logits.shape = (batch_size, sequence_length, num_classes)
```

then:

```python
torch.softmax(logits, dim=-1)
```

applies softmax over:

```text
(batch_size, sequence_length, classes)
                              ↑
                           softmax here
```

The result is that the class probabilities for each sample/position sum to 1.

---

### Important reminder

The correct `dim` is determined by the **meaning of the dimensions**, not by the number of dimensions.

Always check:

1. What does each dimension represent?
2. Which dimension contains the class scores?
3. Apply softmax over that class dimension.